In [4]:
#---
# Rate Limits – understanding, inspecting, and proactively respecting API rate limits.
# Retrying – strategies for automatically retrying failed requests using exponential backoff.
# RPM: requests per minute , TPM: tokens per minute, RPD, TPD, IPM etc.

import os
import random
import time
from getpass import getpass
import requests
from tenacity import retry, stop_after_attempt, wait_random_exponential
from llm_config import groq, MODEL_GROQ, groq_api_key

# Each response exposes headers like x-ratelimit-remaining-requests

In [5]:
header = { 
    'Authorization' : f'Bearer {groq_api_key}',
    'Content-Type'  : 'application/json' }
input_message = [{"role": "user", "content" : "hello"}]
data = {'model' : MODEL_GROQ, 'messages' : input_message, }

response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions", 
    headers = header, 
    json=data) 

# Check if request was successful and headers are present
if response.status_code == 200:
    remaining = response.headers.get('x-ratelimit-remaining-requests', 'Not available')
    print("Remaining requests:", remaining)

    for key, value in response.headers.items():
        if 'ratelimit' in key.lower():
            print(f"{key} : ",f"{value}")
else:
    print(f"Request failed with status code: {response.status_code}")
    print(response.text)

Remaining requests: 999
x-ratelimit-limit-requests :  1000
x-ratelimit-limit-tokens :  8000
x-ratelimit-remaining-requests :  999
x-ratelimit-remaining-tokens :  7597
x-ratelimit-reset-requests :  1m26.4s
x-ratelimit-reset-tokens :  3.022s


In [ ]:
"""
Remaining requests: 999
x-ratelimit-limit-requests :  1000              # Maximum requests allowed in the current limit period
x-ratelimit-limit-tokens :  8000
x-ratelimit-remaining-requests :  999           # Requests you still have available
x-ratelimit-remaining-tokens :  7597
x-ratelimit-reset-requests :  1m26.4s           # When the request limit resets
x-ratelimit-reset-tokens :  3.022s              # When the token limit resets
"""

In [ ]:
# -------------Retry strategies -----------
# When you do hit a rate limit (429), automatic retries with exponential backoff help smooth over spikes.
# Use tenacity to retry on RateLimitError.

from tenacity import retry, stop_after_attempt, wait_random_exponential
@retry(
    reraise=True,
    stop=stop_after_attempt(6),
    wait=wait_random_exponential(min=1, max=60)
)
def create_with_backoff(**kwargs):
    return client.responses.create(**kwargs)

# call with retry logic
resp = create_with_backoff(
    model=MODEL,
    input=[{"role":"user","content":"Tell me a joke."}]
)
print(resp.output_text)
